# cert/v2 · 2026 데이터 확인

`prepro_2026.py` 산출물(`data_2026/`)을 점검하는 노트북입니다. 원본 `data/식품_일반음식점.csv`(2026 인허가 반영)를 v1과 동일한 방식으로 전처리한 결과가 맞는지, 그리고 8종 차트가 쓰는 집계 수치를 확인합니다.

**실행 방법** (cwd = `cert/v2`):
```
uv sync                 # duckdb, pandas 설치
uv run jupyter lab      # 또는 VS Code 에서 이 노트북 열기
```

## 1. 연결 & 테이블 목록

In [1]:
from pathlib import Path
import duckdb, pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.unicode.east_asian_width", True)

DB = Path("data_2026/database.db")          # cert/v2 기준 상대경로
assert DB.exists(), f"{DB} 없음 — 먼저 `python prepro_2026.py` 실행"
con = duckdb.connect(str(DB), read_only=True)
con.execute("SHOW TABLES").df()

,name
0,restaurant_2021
1,restaurant_2022
2,restaurant_2023
3,restaurant_2024
4,restaurant_2025
5,restaurant_2026


## 2. 연도별 행 수 (연도당 10,120~10,200 범위)

In [2]:
tables = [f"restaurant_{y}" for y in range(2021, 2027)]
pd.DataFrame({
    "table": tables,
    "rows": [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables],
})

,table,rows
0,restaurant_2021,10135
1,restaurant_2022,10160
2,restaurant_2023,10184
3,restaurant_2024,10185
4,restaurant_2025,10133
5,restaurant_2026,10148


## 3. 컬럼 스키마 (v1 15개 컬럼과 동일해야 함)

In [3]:
con.execute("DESCRIBE restaurant_2026").df()[["column_name", "column_type"]]

,column_name,column_type
0,개방서비스명,VARCHAR
1,인허가일자,TIMESTAMP_NS
2,폐업일자,VARCHAR
3,영업상태명,VARCHAR
4,소재지,VARCHAR
5,시설명,VARCHAR
6,구분,VARCHAR
7,남성종사자수,DOUBLE
8,여성종사자수,DOUBLE
9,시도,VARCHAR


## 4. 샘플 데이터 (2026)

In [4]:
con.execute("SELECT * FROM restaurant_2026 LIMIT 5").df()

,개방서비스명,인허가일자,폐업일자,영업상태명,소재지,시설명,구분,남성종사자수,여성종사자수,시도,시군구,읍면동,도로명,번지,num
0,일반음식점,2026-08-19,운영중,영업/정상,경상남도 산청군 신안면 둔철산로 575-26,비토애럭셔리글램핑산청점,기타,NaN,NaN,경상남도,산청군,신안면,둔철산로,575-26,1
1,일반음식점,2026-08-19,운영중,영업/정상,서울특별시 송파구 문정로4길 28,컴샵(comshop),일식,NaN,NaN,서울특별시,송파구,문정로4길,,28,1
2,일반음식점,2026-08-19,운영중,영업/정상,경상북도 포항시 남구 오천읍 문덕로 35,슬이네제주해장국,한식,NaN,NaN,경상북도,포항시,남구,오천읍 문덕로,35,1
3,일반음식점,2026-08-19,운영중,영업/정상,경상북도 포항시 남구 오천읍 문덕로 35,신불티날개,식육(숯불구이),NaN,NaN,경상북도,포항시,남구,오천읍 문덕로,35,1
4,일반음식점,2026-08-19,운영중,영업/정상,서울특별시 송파구 충민로 66,분더커피바 현대아울렛 가든파이브점,기타,NaN,NaN,서울특별시,송파구,충민로,,66,1


## 5. 무결성 점검

- **연도일치**: `인허가일자`의 연도가 테이블 연도와 모두 일치
- **num=1**: 집계용 카운트 컬럼이 전부 1
- **시도/구분 결측**: 주소 분해가 안 된(부분주소) 행 수 — 참고용

In [5]:
rows = []
for t in tables:
    y = int(t.split("_")[1])
    d = con.execute(f"SELECT * FROM {t}").df()
    d["인허가일자"] = pd.to_datetime(d["인허가일자"])
    rows.append({
        "table": t,
        "rows": len(d),
        "연도일치": bool((d["인허가일자"].dt.year == y).all()),
        "num=1": bool((d["num"] == 1).all()),
        "시도결측": int(d["시도"].isna().sum()),
        "구분결측": int(d["구분"].isna().sum()),
    })
pd.DataFrame(rows)

,table,rows,연도일치,num=1,시도결측,구분결측
0,restaurant_2021,10135,True,True,136,0
1,restaurant_2022,10160,True,True,228,0
2,restaurant_2023,10184,True,True,169,0
3,restaurant_2024,10185,True,True,271,0
4,restaurant_2025,10133,True,True,443,0
5,restaurant_2026,10148,True,True,649,0


## 6. 차트별 집계 미리보기

아래 쿼리는 `lib/dashboard.py`의 `aggregate()`와 동일합니다 — 8종 차트가 실제로 그리는 수치를 그대로 확인할 수 있습니다. `T` 를 바꾸면 다른 연도로 볼 수 있습니다.

In [6]:
T = "restaurant_2026"   # 확인할 연도 테이블

### 6-1. 시도별 음식점 수 — 막대

In [7]:
con.execute(f"SELECT 시도, SUM(num) AS n FROM {T} WHERE 시도 IS NOT NULL GROUP BY 시도 ORDER BY n DESC").df()

,시도,n
0,서울특별시,2209.0
1,경기도,2141.0
2,인천광역시,541.0
3,전남광주통합특별시,535.0
4,부산광역시,529.0
5,경상북도,481.0
6,경상남도,477.0
7,대구광역시,458.0
8,강원특별자치도,416.0
9,충청남도,409.0


### 6-2. 영업 상태 — 퍼널

In [8]:
con.execute(f"SELECT 영업상태명, SUM(num) AS n FROM {T} WHERE 영업상태명 IS NOT NULL GROUP BY 영업상태명 ORDER BY n DESC").df()

,영업상태명,n
0,영업/정상,8548.0
1,폐업,1600.0


### 6-3. 전체 업종 분포 — 파이 / 트리맵 / 히트맵 소스

In [9]:
con.execute(f"SELECT 구분, SUM(num) AS n FROM {T} WHERE 구분 IS NOT NULL GROUP BY 구분 ORDER BY n DESC").df()

,구분,n
0,한식,3847.0
1,기타,3318.0
2,호프/통닭,563.0
3,경양식,442.0
4,일식,374.0
5,분식,332.0
6,중국식,314.0
7,식육(숯불구이),303.0
8,"외국음식전문점(인도,태국등)",185.0
9,횟집,110.0


### 6-4. 시도 × 구분 교차 — 히트맵/트리맵 소스 (피벗)

In [10]:
bt = con.execute(
    f"SELECT 시도, 구분, SUM(num) AS n FROM {T} "
    "WHERE 시도 IS NOT NULL AND 구분 IS NOT NULL GROUP BY ALL"
).df()
bt.pivot_table(index="구분", columns="시도", values="n", fill_value=0)

시도,강원특별자치도,경기도,경상남도,경상북도,대구광역시,대전광역시,부산광역시,서울특별시,세종특별자치시,울산광역시,인천광역시,전남광주통합특별시,전북특별자치도,제주특별자치도,충청남도,충청북도
구분,,,,,,,,,,,,,,,,
감성주점,1.0,9.0,5.0,3.0,1.0,1.0,0.0,18.0,0.0,0.0,4.0,2.0,1.0,3.0,2.0,0.0
경양식,6.0,81.0,24.0,14.0,20.0,9.0,20.0,141.0,0.0,14.0,26.0,22.0,6.0,23.0,11.0,17.0
기타,141.0,664.0,105.0,103.0,167.0,158.0,226.0,789.0,7.0,45.0,135.0,189.0,98.0,25.0,131.0,71.0
김밥(도시락),1.0,5.0,3.0,2.0,2.0,1.0,2.0,8.0,0.0,1.0,1.0,2.0,1.0,0.0,3.0,6.0
냉면집,0.0,4.0,2.0,1.0,1.0,0.0,2.0,5.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,1.0
라이브카페,0.0,2.0,0.0,1.0,0.0,0.0,1.0,4.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0
복어취급,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
분식,7.0,71.0,18.0,16.0,12.0,6.0,14.0,57.0,2.0,8.0,25.0,20.0,13.0,4.0,21.0,10.0
뷔페식,3.0,12.0,3.0,3.0,1.0,1.0,2.0,7.0,0.0,0.0,0.0,3.0,1.0,2.0,0.0,3.0


### 6-5. 시도 → 영업상태 → 업종 — 다단계 생키 소스 (상위 20)

In [11]:
con.execute(
    f"SELECT 시도, 영업상태명, 구분, SUM(num) AS n FROM {T} "
    "WHERE 시도 IS NOT NULL AND 영업상태명 IS NOT NULL AND 구분 IS NOT NULL "
    "GROUP BY ALL ORDER BY n DESC LIMIT 20"
).df()

,시도,영업상태명,구분,n
0,경기도,영업/정상,한식,778.0
1,서울특별시,영업/정상,한식,747.0
2,서울특별시,영업/정상,기타,587.0
3,경기도,영업/정상,기타,517.0
4,서울특별시,폐업,기타,202.0
5,인천광역시,영업/정상,한식,181.0
6,경상남도,영업/정상,한식,177.0
7,경상북도,영업/정상,한식,176.0
8,전남광주통합특별시,영업/정상,한식,172.0
9,전남광주통합특별시,영업/정상,기타,167.0


### 6-6. 인허가 월별 추세 — 라인

In [12]:
con.execute(
    f"SELECT strftime(인허가일자, '%Y-%m') AS 년월, SUM(num) AS n FROM {T} "
    "GROUP BY 1 ORDER BY 1"
).df()

,년월,n
0,2026-06,2143.0
1,2026-07,5138.0
2,2026-08,2867.0


## 7. 마무리

In [13]:
con.close()